In [5]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT linux -> Test on linux
Mean Squared False Error (MSFE) Loss (Algorithmic Approach), No Synthetic Oversampling
"""
import sys
import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED        = 42
TEST_PROJ   = "linux"
COMBINED_FILE = "../../../../embedding/primevul/codebert/primevul_embedded.jsonl"
EMB_KEY     = "emb"
OUTPUT_DIR  = "results/msfe/linux"

MSFE_BATCH_SIZE = 512

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading 
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# MSFE Loss
# =========================================================
class MSFELoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, predictions, targets):
        predictions = predictions.clamp(self.eps, 1 - self.eps)
        squared_errors = (targets - predictions) ** 2

        pos_mask = (targets == 1)
        neg_mask = (targets == 0)

        fne = squared_errors[pos_mask].mean() if pos_mask.any() else torch.tensor(0.0, device=predictions.device)
        fpe = squared_errors[neg_mask].mean() if neg_mask.any() else torch.tensor(0.0, device=predictions.device)

        msfe = fpe ** 2 + fne ** 2
        return msfe


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                          optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        zero_positive_batches = 0
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            if (y_batch == 1).sum().item() == 0:
                zero_positive_batches += 1

            optimizer.zero_grad()
            predictions = model(x_batch).squeeze()
            loss        = criterion(predictions, y_batch)  
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(x_val).squeeze()
            val_loss  = criterion(val_preds, y_val).item()

        note = f" ({zero_positive_batches} zero-positive batches)" if zero_positive_batches else ""
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}{note}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test,
                                       msfe_criterion, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=MSFE_BATCH_SIZE, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Phase 1: Train on source (all-except-linux)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, msfe_criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (linux)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)
    
    print(f"      [DIAGNOSTIC after Phase 1] y_pred -- min: {y_pred.min():.6f}, "
          f"max: {y_pred.max():.6f}, mean: {y_pred.mean():.6f}, "
          f"pseudo-labels flagged vulnerable: {y_pred_binary.sum()} / {len(y_pred_binary)}")

    print(f"      [DIAGNOSTIC after Phase 1] ...")
    sys.exit("Diagnostic complete -- check printed values above")

    # Phase 2: Fine-tune on source + pseudo-labelled linux
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=MSFE_BATCH_SIZE, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, msfe_criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# MSFE objective for XGBoost 
# =========================================================
def make_xgb_msfe_obj():
    eps = 1e-7
    dz = 1e-4

    def msfe_obj(y_true, y_pred, sample_weight=None):
        y = np.asarray(y_true, dtype=np.float64)
        z = np.asarray(y_pred, dtype=np.float64)

        pos_mask = (y == 1)
        neg_mask = (y == 0)
        n_pos = int(pos_mask.sum())
        n_neg = int(neg_mask.sum())

        def squared_err(z_vec):
            p = 1.0 / (1.0 + np.exp(-z_vec))
            p = np.clip(p, eps, 1 - eps)
            return (y - p) ** 2

        sq_err_base = squared_err(z)
        fpe_base = sq_err_base[neg_mask].mean() if n_neg > 0 else 0.0
        fne_base = sq_err_base[pos_mask].mean() if n_pos > 0 else 0.0
        msfe_base = fpe_base ** 2 + fne_base ** 2

        sq_err_up   = squared_err(z + dz)
        sq_err_down = squared_err(z - dz)

        grad = np.zeros_like(z)
        hess = np.full_like(z, 1e-6)

        if n_pos > 0:
            delta_up_pos   = (sq_err_up[pos_mask]   - sq_err_base[pos_mask])   / n_pos
            delta_down_pos = (sq_err_down[pos_mask] - sq_err_base[pos_mask])   / n_pos
            msfe_up_pos    = fpe_base ** 2 + (fne_base + delta_up_pos) ** 2
            msfe_down_pos  = fpe_base ** 2 + (fne_base + delta_down_pos) ** 2
            grad[pos_mask] = (msfe_up_pos - msfe_down_pos) / (2 * dz)
            hess[pos_mask] = np.clip(
                (msfe_up_pos - 2 * msfe_base + msfe_down_pos) / (dz ** 2), 1e-6, None
            )

        if n_neg > 0:
            delta_up_neg   = (sq_err_up[neg_mask]   - sq_err_base[neg_mask])   / n_neg
            delta_down_neg = (sq_err_down[neg_mask] - sq_err_base[neg_mask])   / n_neg
            msfe_up_neg    = (fpe_base + delta_up_neg) ** 2 + fne_base ** 2
            msfe_down_neg  = (fpe_base + delta_down_neg) ** 2 + fne_base ** 2
            grad[neg_mask] = (msfe_up_neg - msfe_down_neg) / (2 * dz)
            hess[neg_mask] = np.clip(
                (msfe_up_neg - 2 * msfe_base + msfe_down_neg) / (dz ** 2), 1e-6, None
            )

        if sample_weight is not None:
            sw = np.asarray(sample_weight, dtype=np.float64)
            grad = grad * sw
            hess = hess * sw

        return grad, hess

    return msfe_obj


def train_base_model_xgb(x_train, y_train, sample_weights=None):
    msfe_obj = make_xgb_msfe_obj()
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        objective=msfe_obj,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (linux Test Set) - MSFE")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except linux -> Test on linux (MSFE) ===")
    print(f"    Device: {DEVICE}")

    print("\n[1/5] Loading combined embeddings...")
    X_all, y_all, proj_all = load_jsonl(COMBINED_FILE)

    train_mask = proj_all != TEST_PROJ
    train_emb  = X_all[train_mask].astype(np.float32)
    train_lbl  = y_all[train_mask].astype(np.int32)

    test_mask = proj_all == TEST_PROJ
    test_emb  = X_all[test_mask].astype(np.float32)
    test_lbl  = y_all[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check COMBINED_FILE path.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {COMBINED_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_all[train_mask]))}")
    print(f"      Test set (linux)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")
    print(f"      Expected positives/batch (batch_size={MSFE_BATCH_SIZE}): "
          f"{MSFE_BATCH_SIZE * train_lbl.sum() / len(train_lbl):.2f}")

    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    msfe_criterion = MSFELoss()

    print("\n[3/5] Running ensemble iterations...")
    num_iterations       = 1
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network (MSFE)...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb, train_lbl,
            test_emb,
            msfe_criterion,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost (MSFE objective)...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(
            train_emb, train_lbl, sample_weights
        )
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    print("\n[4/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (linux Test Set) - MSFE ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "msfe_no_oversampling",
        "train"            : "all_except_linux",
        "test"             : TEST_PROJ,
        "msfe_batch_size"  : MSFE_BATCH_SIZE,
        "n_train_projects" : int(len(set(proj_all[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "msfe_no_oversampling_linux.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_msfe_no_oversampling_linux.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()


=== PrimeVul | Train on ALL except linux -> Test on linux (MSFE) ===
    Device: mps

[1/5] Loading combined embeddings...
      Training pool             : (170682, 768)  vuln=4852  benign=165830
      Projects in training pool : 753
      Test set (linux)          : (53851, 768)  vuln=1152  benign=52699
      Training imbalance ratio  : 0.0284
      Expected positives/batch (batch_size=512): 14.55

[2/5] Normalizing embeddings...

[3/5] Running ensemble iterations...


Ensemble:   0%|          | 0/1 [00:00<?, ?iter/s]


      [Iteration 1/1] Training neural network (MSFE)...


Iter 1 - Phase 1:   0%|          | 0/50 [00:00<?, ?it/s]

      Iter 1 - Phase 1 | Epoch 1/50 - val_loss: 0.0632
      Iter 1 - Phase 1 | Epoch 2/50 - val_loss: 0.0673
      Iter 1 - Phase 1 | Epoch 3/50 - val_loss: 0.0613
      Iter 1 - Phase 1 | Epoch 4/50 - val_loss: 0.0543
      Iter 1 - Phase 1 | Epoch 5/50 - val_loss: 0.0554
      Iter 1 - Phase 1 | Epoch 6/50 - val_loss: 0.0534
      Iter 1 - Phase 1 | Epoch 7/50 - val_loss: 0.0530
      Iter 1 - Phase 1 | Epoch 8/50 - val_loss: 0.0557
      Iter 1 - Phase 1 | Epoch 9/50 - val_loss: 0.0554
      Iter 1 - Phase 1 | Epoch 10/50 - val_loss: 0.0628
      Iter 1 - Phase 1 | Epoch 11/50 - val_loss: 0.0566
      Iter 1 - Phase 1 | Epoch 12/50 - val_loss: 0.0587
      Iter 1 - Phase 1 | Epoch 13/50 - val_loss: 0.0590
      Iter 1 - Phase 1 | Epoch 14/50 - val_loss: 0.0616
      Iter 1 - Phase 1 | Epoch 15/50 - val_loss: 0.0646
      Iter 1 - Phase 1 | Epoch 16/50 - val_loss: 0.0644
      Iter 1 - Phase 1 | Epoch 17/50 - val_loss: 0.0729
      Iter 1 - Phase 1 | Epoch 18/50 - val_loss: 0.0675
 

SystemExit: Diagnostic complete -- check printed values above